In [1]:
import re
import json
import pandas as pd
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Grocery_and_Gourmet_Food",
    split="full",
    trust_remote_code=True
)

df = pd.DataFrame(dataset).rename(columns={"parent_asin": "asin"})[
    ["asin", "title", "price", "average_rating"]
]
df = df.dropna(subset=["title"])
df = df.drop_duplicates(subset=["asin"])


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/martitesti/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/martitesti/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
/Users/martitesti/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_ids = {
    "ggf-note-000002", "ggf-note-000048", "ggf-note-000013", "ggf-note-000124",
    "ggf-note-000010", "ggf-note-000170", "ggf-note-000001", "ggf-note-000039",
    "ggf-note-000015", "ggf-note-000058", "ggf-note-000022", "ggf-note-000003",
    "ggf-note-000008", "ggf-note-000072", "ggf-note-000005",
}

selected_notes = []
with open("/Users/martitesti/Desktop/uni/tesi/ShopJournal/evaluation_test/ggf_notes_student_v0.1.jsonl", "r") as f:
    for line in f:
        note = json.loads(line)
        if note["note_id"] in selected_ids:
            selected_notes.append(note)

notes_dict = {n["note_id"]: n for n in selected_notes}

queries = [
    {"query_id": "q01", "note_id": "ggf-note-000002", "query": "I want individually wrapped milk chocolate candy"},
    {"query_id": "q02", "note_id": "ggf-note-000048", "query": "I'm looking for organic dark roast whole bean coffee"},
    {"query_id": "q03", "note_id": "ggf-note-000013", "query": "I want a dried fruit and nut gift box, gluten-free"},
    {"query_id": "q04", "note_id": "ggf-note-000124", "query": "I want organic caffeine-free herbal tea bags"},
    {"query_id": "q05", "note_id": "ggf-note-000010", "query": "I need ground coffee for a quick breakfast"},
    {"query_id": "q06", "note_id": "ggf-note-000170", "query": "I need colorful candy sprinkles for cake decorating"},
    {"query_id": "q07", "note_id": "ggf-note-000001", "query": "I'm looking for chicken bone broth, low sodium"},
    {"query_id": "q08", "note_id": "ggf-note-000039", "query": "I want gluten-free vegan protein bars"},
    {"query_id": "q09", "note_id": "ggf-note-000015", "query": "I want organic frozen grilled vegetables"},
    {"query_id": "q10", "note_id": "ggf-note-000058", "query": "I need canned tuna or canned fish for the pantry"},
    {"query_id": "q11", "note_id": "ggf-note-000022", "query": "I need multigrain crackers with seeds"},
    {"query_id": "q12", "note_id": "ggf-note-000003", "query": "I want a dried fruit snack with no added sugar"},
    {"query_id": "q13", "note_id": "ggf-note-000008", "query": "I'm looking for tart cherry juice concentrate"},
    {"query_id": "q14", "note_id": "ggf-note-000072", "query": "I need a ready-to-eat low calorie chicken meal"},
    {"query_id": "q15", "note_id": "ggf-note-000005", "query": "I'm looking for Himalayan pink salt for everyday cooking"},
]

In [3]:
from functions import matchQuery, matchNotes, cueScore, popularityScore, extract_cues, score_query

flags = ["query_only", "query_notes", "query_notes_pop", "query_notes_pop_cue"]

all_rows = []

for q in queries:
    note = notes_dict[q["note_id"]]
    results = score_query(q["query_id"], q["query"], note, df, flags)

    for flag in flags:
        for rank, item in enumerate(results[flag], start=1):
            all_rows.append({
                "query_id": q["query_id"],
                "asin": item["asin"],
                "title": item["title"],
                "query": q["query"],
                "note_id": q["note_id"],
                "note_text": note["note_text"],
                "price": item["price"] if item["price"] is not None else "",
                "source": flag,
                "rank": rank,
                "score": item["score"],
                "retrieval_version": "v1",
            })

print(f"Totale righe generate: {len(all_rows)}")

Totale righe generate: 600


In [4]:
import csv

fieldnames = [
    "query_id", "asin", "title", "query", "note_id", "note_text",
    "price", "source", "rank", "score", "retrieval_version"
]

with open("system_candidates.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print("File system_candidates.csv salvato")

File system_candidates.csv salvato


In [5]:
qrels = pd.read_csv("qrels_v1.csv")
candidates = pd.read_csv("system_candidates.csv")

print("Esempio ASIN in qrels:", qrels["asin"].iloc[0])
print("Esempio ASIN in system_candidates:", candidates["asin"].iloc[0])

# controllo overlap manuale per query q01
q01_qrels = set(qrels[qrels["query_id"] == "q01"]["asin"])
q01_cand = set(candidates[candidates["query_id"] == "q01"]["asin"])
print("Overlap ASIN per q01:", q01_qrels & q01_cand)

Esempio ASIN in qrels: B09JJZ6692
Esempio ASIN in system_candidates: B09L8MCPJR
Overlap ASIN per q01: set()


In [ ]:
from functions import get_query_words, prefilter_candidates
for q in queries:
    if q["query_id"] < "q06":
        continue
    note = notes_dict[q["note_id"]]
    query_words = get_query_words(q["query"])
    note_terms = note["distinctive_terms"]
    candidates = prefilter_candidates(query_words, note_terms, df)
    print(f"{q['query_id']} | query_words={query_words} | note_terms={note_terms} | candidati dopo pre-filtro={len(candidates)}")

NameError: name 'get_query_words' is not defined